In [12]:
import kagglehub

# Download latest version
books = kagglehub.dataset_download("arashnic/book-recommendation-dataset", path='Books.csv')

print("Path to dataset files:", books)

Path to dataset files: /kaggle/input/book-recommendation-dataset/Books.csv


In [13]:
import kagglehub

# Download latest version
ratings = kagglehub.dataset_download("arashnic/book-recommendation-dataset",path='Ratings.csv')

print("Path to dataset files:", ratings)

Path to dataset files: /kaggle/input/book-recommendation-dataset/Ratings.csv


In [14]:
# load datasets
import pandas as pd
books = pd.read_csv("/kaggle/input/book-recommendation-dataset/Books.csv")
ratings = pd.read_csv('/kaggle/input/book-recommendation-dataset/Ratings.csv')
books.head()
ratings.head()

<ipython-input-14-8c8b1e10bd78>:3: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  books = pd.read_csv("/kaggle/input/book-recommendation-dataset/Books.csv")


,User-ID,ISBN,Book-Rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6


In [15]:
# Merging both dataset
df = pd.merge(books, ratings, on='ISBN', how='left')
df.head()

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L,User-ID,Book-Rating
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,2.0,0.0
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,8.0,5.0
2,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,11400.0,0.0
3,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,11676.0,8.0
4,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,41385.0,0.0


In [16]:
df.dropna(inplace=True)
df = df.sample(n=20000, random_state=42).reset_index(drop=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ISBN                 20000 non-null  object 
 1   Book-Title           20000 non-null  object 
 2   Book-Author          20000 non-null  object 
 3   Year-Of-Publication  20000 non-null  object 
 4   Publisher            20000 non-null  object 
 5   Image-URL-S          20000 non-null  object 
 6   Image-URL-M          20000 non-null  object 
 7   Image-URL-L          20000 non-null  object 
 8   User-ID              20000 non-null  float64
 9   Book-Rating          20000 non-null  float64
dtypes: float64(2), object(8)
memory usage: 1.5+ MB


**Popularity Base Recommendation System**
To implement a popularity-based recommendation system using the dataset you provided, the idea is to recommend the most popular books based on the number of ratings or the average rating score. Here’s how you can do this:

Step 1: Group the dataset by Book-Title and calculate the average rating and the number of ratings for each book.

Step 2: Sort the books based on either the average rating or the number of ratings (whichever you define as "popularity").

Step 3: Recommend books that have the highest popularity score.

In [17]:
# Step 1: Calculate average ratings and the number of ratings
book_ratings = df.groupby('Book-Title').agg({'Book-Rating': ['mean', 'count']}).reset_index()

# Step 2: Sort the books based on the count of ratings (popularity) or average rating
book_ratings.columns = ['Book-Title', 'Avg-Rating', 'Num-Ratings']
book_ratings_sorted = book_ratings.sort_values(by=['Num-Ratings', 'Avg-Rating'], ascending=False)

# Step 3: Recommend the top N popular books
top_books = book_ratings_sorted.head(10)  # Change N to the number of books you want to recommend

# Step 4: Merge with the original dataset to include Author, Publisher, and Image-URL-M
top_books_final = pd.merge(top_books, df[['Book-Title','Image-URL-M']], on='Book-Title', how='left')

# Step 5: Drop duplicate rows based on Book-Title
top_books_final = top_books_final.drop_duplicates(subset=['Book-Title'])

# Display the top N recommended books with the additional columns
top_books_final


,Book-Title,Avg-Rating,Num-Ratings,Image-URL-M
0,Wild Animus,1.063492,63,http://images.amazon.com/images/P/0971880107.0...
63,The Lovely Bones: A Novel,4.000000,29,http://images.amazon.com/images/P/0316666343.0...
92,Divine Secrets of the Ya-Ya Sisterhood: A Novel,4.181818,22,http://images.amazon.com/images/P/0060928336.0...
114,The Da Vinci Code,4.045455,22,http://images.amazon.com/images/P/0385504209.0...
136,The Nanny Diaries: A Novel,3.952381,21,http://images.amazon.com/images/P/0312291639.0...
157,Summer Sisters,3.294118,17,http://images.amazon.com/images/P/0440226430.0...
174,The Five People You Meet in Heaven,5.062500,16,http://images.amazon.com/images/P/0786868716.0...
190,Suzanne's Diary for Nicholas,4.062500,16,http://images.amazon.com/images/P/0446679593.0...
206,Me Talk Pretty One Day,3.800000,15,http://images.amazon.com/images/P/0316777722.0...
221,A Painted House,2.533333,15,http://images.amazon.com/images/P/038550120X.0...


**Search Base Recommendation System**

In [19]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# Drop duplicates based on 'Book-Title'
df = df.drop_duplicates(subset=['Book-Title']).reset_index(drop=True)

tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(df['Book-Title'])  # Recompute TF-IDF after dropping duplicates

In [20]:
from sklearn.metrics.pairwise import cosine_similarity

def search_based_recommendation(book_title, df,tfidf, tfidf_matrix, top_n=10):
    """
    Recommend books based on similarity of titles using TF-IDF and cosine similarity.

    Parameters:
        book_title (str): The title of the book to search for recommendations.
        df (pd.DataFrame): The dataset containing book information.
        tfidf_matrix (sparse matrix): Precomputed TF-IDF matrix for book titles.
        top_n (int): Number of recommendations to return.

    Returns:
        pd.DataFrame: Recommended books (title, rating, image URL).
    """


    # Transform the input title to match TF-IDF dimensions
    query_vector = tfidf.transform([book_title])

    # Compute cosine similarity between the query and all book titles
    cosine_sim = cosine_similarity(query_vector, tfidf_matrix).flatten()

    # Get indices of the top N most similar books (excluding the input book itself)
    sim_scores = list(enumerate(cosine_sim))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)  # Sort by similarity
    sim_scores = sim_scores[1:top_n + 1]  # Exclude the input book itself
     # Get the indices of the recommended books
    book_indices = [i[0] for i in sim_scores]

    # Return the relevant columns for the recommended books
    return df[['Book-Title', 'Book-Rating', 'Image-URL-M']].iloc[book_indices].reset_index(drop=True)



In [22]:
# Example usage
book_to_search = "The Hobbit and The Lord of the Rings	"  # Replace with the title you want to search
recommended_books = search_based_recommendation(book_to_search, df,tfidf, tfidf_matrix)

recommended_books

,Book-Title,Book-Rating,Image-URL-M
0,The Lord of the Rings,0.0,http://images.amazon.com/images/P/0618153969.0...
1,The Hobbit : The Enchanting Prelude to The Lor...,10.0,http://images.amazon.com/images/P/0345339681.0...
2,The Lord of the Rings: The QPB Companion to th...,0.0,http://images.amazon.com/images/P/0965307883.0...
3,The Lord of the Rings: A Location Guidebook (L...,9.0,http://images.amazon.com/images/P/1869504917.0...
4,The Hobbit,10.0,http://images.amazon.com/images/P/0345917421.0...
5,"The Return of the King (The Lord of the Rings,...",0.0,http://images.amazon.com/images/P/0345339738.0...
6,The Fellowship of the Ring (Lord of the Rings ...,7.0,http://images.amazon.com/images/P/0345296052.0...
7,The Fellowship of the Ring (The Lord of the Ri...,10.0,http://images.amazon.com/images/P/0618129030.0...
8,"The Two Towers (The Lord of the Rings, Part 2)",0.0,http://images.amazon.com/images/P/0345339711.0...
9,Lo Hobbit / The Hobbit,0.0,http://images.amazon.com/images/P/8845906884.0...
